Implementing Multilevel Monte Carlo of the Finite-Element Poisson Problem 
Melayne and Neal
For PV's class Math 563 in Spring 2026
Update for May 6.

In [15]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [16]:
# Domain is unit square, create initial mesh
N = 64 
mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
mesh.nv, mesh.ne
Draw(mesh);

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [17]:
# Working with H1-conforming piecewise linear finite elements
# boundary conditions are Dirichlet on the left and default=Neumann on the other 3 sides
fes = H1(mesh, order=1, dirichlet="left")
fes.ndof

# Set test and trial spaces
u = fes.TrialFunction()
phi = fes.TestFunction()

# The solution will be stored as a grid function
gfu = GridFunction(fes)
gfu.Set(1, BND)

In [18]:
# Coefficient function for K(x,y)
Kxy = CoefficientFunction(1+x*x+y*y)
# Kxy = CoefficientFunction(3)
sigma = 0.5

# Bilinear form
a = BilinearForm(fes)
a += Kxy*grad(u)*grad(phi)*dx+sigma*sigma*u*phi*dx
a.Assemble()

# Right hand side
f = LinearForm(fes)
f.Assemble()


In [19]:
print(a.mat)
# print(f.vec)

Row 0:   0: 1.00006   4: -0.500025   255: -0.500025
Row 1:   1: 1.68998   66: -0.598353   67: -0.599954   318: -0.491658
Row 2:   2: 2.97923   129: -1.48961   130: -1.48961
Row 3:   3: 1.68896   192: -0.599152   193: -0.594513   443: -0.495283
Row 4:   0: -0.500025   4: 1.89621   5: -0.522629   255: -0.161394   256: -0.712124
Row 5:   4: -0.522629   5: 1.87707   6: -0.336284   256: -0.204683   257: -0.813435
Row 6:   5: -0.336284   6: 1.74719   7: -0.27919   257: -0.474185   258: -0.6575
Row 7:   6: -0.27919   7: 1.73999   8: -0.266081   258: -0.591456   259: -0.603241
Row 8:   7: -0.266081   8: 1.74463   9: -0.267626   259: -0.626007   260: -0.584892
Row 9:   8: -0.267626   9: 1.74912   10: -0.271602   260: -0.627368   261: -0.582503
Row 10:   9: -0.271602   10: 1.75419   11: -0.276682   261: -0.623998   262: -0.581887
Row 11:   10: -0.276682   11: 1.76011   12: -0.28288   262: -0.619812   263: -0.580713
Row 12:   11: -0.28288   12: 1.76691   13: -0.289298   263: -0.613695   264: -0.5

In [20]:
Draw(Kxy, mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [21]:
Draw(gfu)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [ ]:
# Solve using CG
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=f, gf=gfu, pre=c)

CG iteration 1, residual = 5.808946276084119     
CG iteration 2, residual = 2.9146261374940554     
CG iteration 3, residual = 1.9733193425747573     
CG iteration 4, residual = 1.5022746042146324     
CG iteration 5, residual = 1.2472459357087808     
CG iteration 6, residual = 1.0530915369900873     
CG iteration 7, residual = 0.8903467122981884     
CG iteration 8, residual = 0.7801549878776666     
CG iteration 9, residual = 0.700421895460815     
CG iteration 10, residual = 0.6336776295319169     
CG iteration 11, residual = 0.575069811732648     
CG iteration 12, residual = 0.5239805745088854     
CG iteration 13, residual = 0.48562810534738676     
CG iteration 14, residual = 0.45688200678020374     
CG iteration 15, residual = 0.4246631693132494     
CG iteration 16, residual = 0.3928232225651809     
CG iteration 17, residual = 0.3702824711907346     
CG iteration 18, residual = 0.35217838068294965     
CG iteration 19, residual = 0.33586262453717075     
CG iteration 20, res

In [ ]:
# Plot the solution
Draw(gfu)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [24]:
# Test a nice u and find f then work back to it
# Given u(x,y)=1-3x^2+2x^3 
plant = CoefficientFunction(1-3*x*x+2*x*x*x)
Draw(plant, mesh)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [25]:
# Manually calculate the right hand side f for the u above "plant"
fakef = 25/4 - 12*x + 69/4 * x**2 - 47/2 * x**3 + 6 * y**2 - 12 * x * y**2
Draw(fakef,mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [26]:
# Solve the problem with this new right hand side
newf = LinearForm(fes)
newf += fakef*phi*dx
newf.Assemble()

gfu2 = GridFunction(fes)
gfu2.Set(1, BND)
c = Preconditioner(a,"local")
c.Update()
solvers.BVP(bf=a, lf=newf, gf=gfu2, pre=c)

Draw(gfu2)

CG iteration 1, residual = 5.815226427924762     
CG iteration 2, residual = 2.9239433182582     
CG iteration 3, residual = 1.986329209229491     
CG iteration 4, residual = 1.5190078704731553     
CG iteration 5, residual = 1.2680248586752036     
CG iteration 6, residual = 1.0774797668786065     
CG iteration 7, residual = 0.9178495004968749     
CG iteration 8, residual = 0.8111661701679197     
CG iteration 9, residual = 0.7349679335273317     
CG iteration 10, residual = 0.671468991579208     
CG iteration 11, residual = 0.6156904911798654     
CG iteration 12, residual = 0.5670999087097814     
CG iteration 13, residual = 0.5315549256563473     
CG iteration 14, residual = 0.5059394447414967     
CG iteration 15, residual = 0.4759288440153664     
CG iteration 16, residual = 0.4459197045384837     
CG iteration 17, residual = 0.4264515973708109     
CG iteration 18, residual = 0.41268106745854455     
CG iteration 19, residual = 0.4002980881736891     
CG iteration 20, residual 

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene